# 디킨스 실습실: 『Sketches by Boz』를 코드로 읽기
# Dickens Lab: reading *Sketches by Boz* with code

**선택 실습 · 제출 없음 / Optional · No submission**
SKKU 특강 "Vibe Coding in the Humanities" (2026-10-16) 후속 실습 / follow-up lab

- **무엇을 / What:** 교수님 슬라이드의 과제 목록 6가지를 하나씩 해 봅니다. 특강에서 끝까지 하지 못한 부분입니다.
  The six tasks on the professor's task list, one at a time — the parts the lecture did not finish.
- **각 과제의 구성 / Each task has:** (a) 연구 질문 한 줄 / a one-line research question · (b) AI에 붙여 넣을 프롬프트 / a prompt to paste into an AI · (c) 실행되는 참고 코드 / working reference code · (d) "나는 무엇을 정했나?" / "What did you decide?"
- **실행 / Run:** 위에서부터 셀을 차례로 실행하세요(셀 왼쪽 ▶ 또는 Shift+Enter). **섹션 0은 꼭 먼저.** / Run cells from the top (▶ or Shift+Enter). **Run section 0 first.**
- **설치할 것 없음 / Nothing to install:** Colab에 이미 있는 pandas, nltk, scikit-learn, matplotlib만 씁니다. / Only libraries Colab already has.

| # | 과제 / Task | 시간 / Time |
|---|---|---|
| 0 | 준비: 텍스트 내려받기 / Setup: download | 5분 / min |
| 1 | 머리말·꼬리말·메타데이터 제거, 56편 나누기 / Strip header, footer, metadata; split into 56 sketches | 10 |
| 2 | 불용어 + 화이트리스트, 그리고 '기능어는 장르 신호' / Stopwords + whitelist; function words as genre signals | 15 |
| 3 | 장소 목록과 빈도 / Places and their frequencies | 10 |
| 4 | 거리 목록과 빈도 (Monmouth-street 문제) / Streets and their frequencies | 10 |
| 5 | 연어(collocation) / Collocations | 10 |
| 6 | 토픽 모델링 / Topic modelling | 15 |

아무 데서나 멈춰도 됩니다. 한 과제만 해도 충분합니다. / Stop anywhere. One task is already worthwhile.

## 이 노트북에서 바이브 코딩하는 법 / How to vibe-code in this notebook

1. 각 과제의 **프롬프트 상자**를 복사해 아무 AI(ChatGPT, Gemini, Claude, Colab의 AI 기능 등)에 붙여 넣습니다.
   Copy a task's **prompt box** into any AI chat (ChatGPT, Gemini, Claude, Colab's built-in AI, …).
2. AI가 준 코드를 새 코드 셀(**+ Code**)에 붙여 넣고 실행합니다. / Paste the code into a new cell (**+ Code**) and run it.
3. **고치기 / Fix it:** 오류가 나면 오류 메시지 전체를 복사해 "이 오류를 고쳐 줘"와 함께 보냅니다. 결과가 이상하면 무엇이 이상한지 구체적으로 말합니다(예: "Monmouth-street를 못 찾았어").
   If it fails, paste the whole error back with "fix this error". If the result looks wrong, say exactly what is wrong (e.g. "it missed Monmouth-street").
4. 내 결과를 그 아래 **참고 코드(Reference)** 결과와 비교합니다. 숫자가 다르면, *왜* 다른지가 곧 연구 질문입니다.
   Compare with the **Reference** cell. If the numbers differ, *why* they differ is your research question.

> 참고 코드는 정답이 아니라 비교 기준입니다. 숫자가 달라도 틀린 것이 아닐 수 있습니다. '단어'나 '거리'를 다르게 정의했을 수 있으니까요.
> The reference is a baseline, not the answer key. Different numbers can mean a different (and defensible) definition of a "word" or a "street".

**AI 대화를 시작할 때 한 번 붙여 넣는 맥락 / Paste this once at the start of your AI chat**

```text
I am a humanities student and a beginner in Python. I work in Google Colab.
Data: Charles Dickens, Sketches by Boz, Project Gutenberg eBook #882 (1903 Chapman & Hall edition).
Give me one code cell at a time. Use only the Python standard library, pandas, nltk, scikit-learn
and matplotlib (Colab already has them). Add a one-line comment for each step.
Do not guess results: I will run the code and paste the output back to you.
```

```text
나는 파이썬 초보인 인문학 전공 학생이고 Google Colab에서 작업해.
자료: 찰스 디킨스 『Sketches by Boz』, Project Gutenberg #882 (1903년 Chapman & Hall판).
코드 셀은 한 번에 하나씩 줘. 파이썬 표준 라이브러리, pandas, nltk, scikit-learn, matplotlib만 써(Colab에 이미 있어).
단계마다 한 줄 주석을 달아 줘. 결과를 추측하지 마. 내가 실행해서 출력을 붙여 넣을게.
```

**개인정보 / Privacy:** AI에 이름·학번·연락처 같은 개인정보나 남의 글을 붙여 넣지 마세요. 이 실습은 저작권이 끝난 공개 텍스트만 씁니다.
Do not paste personal data (name, student ID, contacts) or other people's writing into an AI. This lab uses only a public-domain text.

---
## 0. 준비: 텍스트 내려받기 / Setup: download the text

- 출처 / Source: Project Gutenberg #882, *Sketches by Boz* (1903 Chapman & Hall 판). 인터넷 요청은 **한 번만** 보냅니다. / We send **one** request.
- 파일은 Colab 세션 저장소에 저장되고, 세션이 끝나면 사라집니다. / The file lives in the Colab session and disappears when the session ends.

In [ ]:
# 0-1. 텍스트 내려받기 (한 번만 요청) / download the text (one request)
import os, re, math, time, collections, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://www.gutenberg.org/cache/epub/882/pg882.txt"
FILE = "pg882.txt"
if not os.path.exists(FILE):                      # 이미 받았으면 다시 받지 않음 / skip if already here
    urllib.request.urlretrieve(URL, FILE)
with open(FILE, encoding="utf-8-sig") as f:
    raw = f.read().replace("\r\n", "\n")      # BOM과 Windows 줄바꿈 정리
print(f"{FILE}: {len(raw):,} characters, {raw.count(chr(10)):,} lines")
print("-" * 60)
print(raw[:600])

In [ ]:
# 0-2. NLTK 불용어 목록 준비 / get the NLTK stopword list
import nltk
try:
    from nltk.corpus import stopwords
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
print("NLTK English stopwords:", len(stopwords.words("english")), "words, e.g.", stopwords.words("english")[:12])

---
## 1. 머리말·꼬리말·메타데이터 제거 / Remove the Gutenberg header, footer and metadata

**연구 질문 / Question:** 이 파일에서 디킨스의 『Sketches by Boz』는 정확히 어디서 시작해 어디서 끝나는가? / Where exactly does *Sketches by Boz* begin and end in this file?

**왜 56편인가 / Why 56, not 87?** 1903년판 파일에는 원래 『Sketches by Boz』(1839년 완결판, 4부 56편: Our Parish · Scenes · Characters · Tales) 뒤에 별개 작품이 함께 묶여 있습니다: *Sketches of Young Gentlemen*(13), *Sketches of Young Couples*(12), *The Mudfog and Other Sketches*(6). 모두 나누면 87편이 되지만, 오늘은 원래 56편만 씁니다.
The 1903 file bundles three later works after the original four sections (56 sketches). Splitting everything gives 87 pieces; we keep the original 56.

**프롬프트 / Prompt (EN)**
```text
pg882.txt (already downloaded in Colab) is Project Gutenberg #882, Sketches by Boz (1903 edition).
Write Python that
1) removes the Project Gutenberg header and footer (everything up to the "*** START OF" line,
   and everything from the "*** END OF" line),
2) skips the front matter (title page, PREFACE) so the text starts at the section heading "OUR PARISH",
3) splits the text into sketches at headings like "CHAPTER I—THE BEADLE..." and remembers the
   section (OUR PARISH, SCENES, CHARACTERS, TALES),
4) stops at the line "SKETCHES OF YOUNG GENTLEMEN", because the 1903 edition bundles other works after it,
5) builds a pandas DataFrame named boz with columns section, title, words, text, and prints
   the number of sketches and words per section.
```
**프롬프트 / Prompt (KO)**
```text
pg882.txt(Colab에 이미 내려받음)는 Project Gutenberg #882 『Sketches by Boz』 1903년판이야.
파이썬 코드를 써 줘.
1) Project Gutenberg 머리말과 꼬리말을 지워("*** START OF" 줄까지, "*** END OF" 줄부터),
2) 제목 페이지와 PREFACE를 건너뛰어 본문이 "OUR PARISH" 제목에서 시작하게 하고,
3) "CHAPTER I—THE BEADLE..." 같은 제목 줄에서 작품을 나누고, 어느 부(OUR PARISH, SCENES, CHARACTERS, TALES)인지 기록해.
4) "SKETCHES OF YOUNG GENTLEMEN" 줄에서 멈춰. 1903년판은 그 뒤에 다른 작품을 함께 묶었어.
5) section, title, words, text 열이 있는 pandas DataFrame boz를 만들고, 부별 작품 수와 단어 수를 출력해.
```
**고치기 프롬프트 / Fix-it prompt:** `I got 0 sketches. Print the first 5 lines that start with "CHAPTER" and check which dash character is used (— or -). Fix the regular expression.`

In [ ]:
# 1-1. Gutenberg 머리말·꼬리말 제거 / strip the Gutenberg header and footer
start = re.search(r"\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", raw)
end = re.search(r"\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK", raw)
body = raw[start.end():end.start()]
print(f"header removed: {start.end():,} chars | footer (license) removed: {len(raw) - end.start():,} chars")

# 1-2. 제목 페이지·PREFACE를 건너뛰고 56편으로 나누기 / skip the front matter, split into sketches
SECTIONS = ["OUR PARISH", "SCENES", "CHARACTERS", "TALES"]   # 원래 Sketches by Boz의 네 부분
BUNDLED = "SKETCHES OF YOUNG GENTLEMEN"                       # 1903년판은 여기부터 다른 작품을 묶음
CHAPTER = re.compile(r"^CHAPTER ([IVXL]+)—(.+)$")              # 'CHAPTER I—THE BEADLE...' (긴 대시 —)

rows, section, rest = [], None, ""
lines = body.split("\n")
for i, line in enumerate(lines):
    s = line.strip()
    if s == BUNDLED:                       # 여기서 멈춤 / stop here
        rest = "\n".join(lines[i:])
        break
    if s in SECTIONS:                      # 부 제목 / section heading
        section = s
        continue
    m = CHAPTER.match(s)
    if section and m:                      # 새 작품 시작 / a new sketch begins
        title = m.group(2).title().replace("’S", "’s")
        rows.append({"section": section, "title": title, "lines": []})
    elif rows:                             # 본문 줄 / body line
        rows[-1]["lines"].append(line)

WORD = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)*")
boz = pd.DataFrame([{"section": r["section"], "title": r["title"],
                     "text": "\n".join(r["lines"]).strip()} for r in rows])
boz["words"] = boz.text.apply(lambda t: len(WORD.findall(t)))
boz = boz[["section", "title", "words", "text"]]

print(f"\nSketches by Boz: {len(boz)} sketches, {boz.words.sum():,} words")
print(boz.groupby("section", sort=False).agg(sketches=("title", "size"), words=("words", "sum")).to_string())
print(f"\nNOT used — other works bundled in the 1903 edition: {len(WORD.findall(rest)):,} words")
s = boz.sort_values("words")
print("shortest:", list(zip(s.title[:3], s.words[:3])))
print("longest: ", list(zip(s.title[-3:], s.words[-3:])))
boz.to_csv("boz_56.csv", index=False)       # 왼쪽 폴더 아이콘에서 내려받을 수 있음 / see the Files panel

In [ ]:
# 1-3. 확인 / sanity checks
for t in ["The Streets—Morning", "The Streets—Night"]:
    row = boz[boz.title == t].iloc[0]
    print(f"{t}: {row.words:,} words | starts: {row.text[:60]!r}")
print("\ncurly apostrophes ’ in the body:", body.count("’"), "| straight ' :", body.count("'"))
print("example:", re.findall(r"\b\w+’\w+\b", boz.text[0])[:6])

**무엇을 정했나요? / What did you decide?**
- **판본과 범위 / Edition and scope:** 1903년판에서 원래 4부 56편만 썼습니다. 87편을 모두 쓰면 1840년대에 따로 나온 작품까지 'Boz'로 분석하게 됩니다. / We kept 56 of 87 pieces. Using all 87 would mix in works published separately.
- **PREFACE(서문)는?** 디킨스가 썼지만 스케치는 아닙니다. 빼는 것이 맞을까요? / Dickens wrote it, but it is not a sketch.
- **곡선 따옴표 / Curly quotes:** 본문은 `don’t`처럼 곡선 아포스트로피(’)를 씁니다. 불용어 목록의 `don't`(직선 ')와 다르니, 과제 2부터는 바꿔서 셉니다. / The text uses ’; stopword lists use '. We normalise from Task 2 on.
- 이 결정을 메모해 두세요. 연구노트의 '선택' 항목이 됩니다. / Write these decisions down — they belong in your research notes.

---
## 2. 불용어 제거와 화이트리스트 / Stopwords and a whitelist

**연구 질문 / Question:** 불용어를 지우면 Boz의 런던에서 무엇이 보이고, 무엇이 사라지는가? / When we remove stopwords, what appears — and what disappears?

**프롬프트 / Prompt (EN)**
```text
Use the DataFrame boz (column text) from the previous cell.
1) Lowercase the text, replace curly apostrophes ’ with ', and split it into words with a regular
   expression (keep words like o'clock and don't as one word).
2) Show three top-20 lists side by side: (A) nothing removed; (B) NLTK English stopwords and
   words shorter than 3 letters removed; (C) the same as B, but always keep the whitelist words
   mr, mrs, miss, sir, lord.
3) Without removing any stopwords, compare the sections SCENES and TALES: which words are much
   more frequent in one than in the other? Use a weighted log-odds ratio (z-score) and show the
   top 12 words for each side, with their rate per 1,000 words and whether they are NLTK stopwords.
```
**프롬프트 / Prompt (KO)**
```text
앞 셀의 DataFrame boz(text 열)를 써.
1) 소문자로 바꾸고 곡선 아포스트로피 ’를 '로 바꾼 뒤, 정규식으로 단어를 나눠(o'clock, don't는 한 단어).
2) 상위 20개 단어 목록 세 개를 나란히 보여 줘: (A) 아무것도 안 지움, (B) NLTK 영어 불용어와 3글자 미만 단어 제거,
   (C) B와 같지만 화이트리스트 mr, mrs, miss, sir, lord는 항상 남김.
3) 불용어를 지우지 않은 채로 SCENES와 TALES 부를 비교해, 한쪽에서 훨씬 많이 쓰인 단어를 찾아 줘.
   가중 로그오즈비(z 점수)로 양쪽 상위 12개를 1천 단어당 빈도, NLTK 불용어 여부와 함께 보여 줘.
```
**고치기 프롬프트 / Fix-it prompt:** `In list B, "mr" disappeared even though it is not in the NLTK list. Why? Fix it so the whitelist words always stay.`

In [ ]:
# 2-1. 단어로 나누기 + 세 가지 Top 20 / tokenise and compare three top-20 lists
def norm(text):
    return text.replace("’", "'").replace("‘", "'")       # 곡선 → 직선 따옴표 / curly → straight
TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")                  # o'clock, don't = 한 단어 / one word
def tokens(text):
    return TOKEN.findall(norm(text).lower())

all_words = [w for t in boz.text for w in tokens(t)]
NLTK_SW = set(stopwords.words("english"))
WHITELIST = {"mr", "mrs", "miss", "sir", "lord"}            # 호칭은 항상 남김 / always keep titles

A = collections.Counter(all_words)
B = collections.Counter(w for w in all_words if w not in NLTK_SW and len(w) >= 3)
C = collections.Counter(w for w in all_words if (w not in NLTK_SW and len(w) >= 3) or w in WHITELIST)

def top(counter, n=20):
    return [f"{w} ({c:,})" for w, c in counter.most_common(n)]
print(f"{len(all_words):,} word tokens, {len(A):,} different words\n")
print(pd.DataFrame({"A: nothing removed": top(A),
                    "B: NLTK stopwords + <3 letters": top(B),
                    "C: B + whitelist": top(C)}, index=range(1, 21)).to_string())
print("\n'mr' count  A:", A["mr"], "| B:", B["mr"], "| C:", C["mr"])

In [ ]:
# 2-2. 내 불용어 추가 / add your own stopwords  — 목록을 바꿔 다시 실행해 보세요 / edit and re-run
MY_STOPWORDS = {"said", "one", "would", "could", "upon", "little", "much", "two"}
D = collections.Counter({w: c for w, c in C.items() if w not in MY_STOPWORDS})
print(pd.DataFrame({"C: NLTK + whitelist": top(C), "D: C + MY_STOPWORDS": top(D)}, index=range(1, 21)).to_string())

### 2-3. 지운 단어가 장르 신호라면? / What if the deleted words are the genre signal?

특강에서 본 발견: **불용어를 지우지 않고** 부(section)끼리 비교하면, 관찰기인 *Scenes*에서는 `is, are, we`가, 이야기인 *Tales*에서는 `I, said, was`가 두드러집니다. 현재형·집단 화자(we) 대 과거형·대화(said). 보통 가장 먼저 지우는 단어들입니다.
The lecture's finding: compare sections **without** removing stopwords, and *Scenes* (observation) leans on `is, are, we` while *Tales* (stories) leans on `I, said, was` — present tense and a collective narrator vs past tense and dialogue. These are exactly the words we usually delete first.

In [ ]:
# 2-3. Scenes vs Tales — 불용어를 지우지 않고 비교 / compare without removing stopwords
# weighted log-odds with an informative prior (Monroe, Colaresi & Quinn 2008)
scenes = collections.Counter(w for t in boz[boz.section == "SCENES"].text for w in tokens(t))
tales  = collections.Counter(w for t in boz[boz.section == "TALES"].text for w in tokens(t))
n1, n2 = sum(scenes.values()), sum(tales.values())
prior = scenes + tales
a0 = sum(prior.values())
rows = []
for w, a in prior.items():
    if a < 30:                                   # 너무 드문 단어는 제외 / skip rare words
        continue
    l1 = math.log((scenes[w] + a) / (n1 + a0 - scenes[w] - a))
    l2 = math.log((tales[w] + a) / (n2 + a0 - tales[w] - a))
    z = (l1 - l2) / math.sqrt(1 / (scenes[w] + a) + 1 / (tales[w] + a))
    rows.append({"word": w, "Scenes /1k": round(1000 * scenes[w] / n1, 2),
                 "Tales /1k": round(1000 * tales[w] / n2, 2), "z": round(z, 1),
                 "NLTK stopword?": "yes" if w in NLTK_SW else ""})
lo = pd.DataFrame(rows).sort_values("z", ascending=False)
print(f"Scenes {n1:,} words | Tales {n2:,} words\n")
print("More in SCENES (observation):")
print(lo.head(12).to_string(index=False))
print("\nMore in TALES (stories):")
print(lo.tail(12).iloc[::-1].to_string(index=False))
top24 = pd.concat([lo.head(12), lo.tail(12)])
print(f"\n{(top24['NLTK stopword?'] == 'yes').sum()} of these 24 signal words are NLTK stopwords.")

**무엇을 정했나요? / What did you decide?**
- **화이트리스트 / Whitelist:** `mr`는 NLTK 불용어가 아닌데 'B'에서 사라졌습니다. 3글자 미만 규칙 때문입니다. 호칭(mr, mrs, miss)은 '이름 있는 인물이 등장하는 장면'의 신호라서 남겼습니다. / `mr` vanished because of the 3-letter rule, not the stopword list.
- **said를 지울까? / Delete "said"?** 내용(무엇에 대해 쓰나)을 보려면 지우고, 문체(어떻게 쓰나: 대화가 많은 이야기인가)를 보려면 남깁니다. / Delete it for *content*; keep it for *style*.
- **결론 / Takeaway:** 불용어 제거는 '청소'가 아니라 **연구 선택**입니다. 기능어(function words)는 문체 분석(stylometry, 저자 판별)의 핵심 재료이기도 합니다. / Stopword removal is a **research choice**, not cleaning; function words are the raw material of stylometry.
- 어떤 단어를 `MY_STOPWORDS`에 넣었고, 왜? / Which words did you add, and why?

---
## 3. 언급된 장소와 빈도 / Places mentioned, with frequencies

**연구 질문 / Question:** Boz는 어떤 장소를 얼마나 자주, 몇 편에 걸쳐 언급하는가? / Which places does Boz mention, how often, and across how many sketches?

방법은 두 단계입니다. (1) 규칙으로 **후보**를 뽑고 사람이 훑어본 뒤, (2) 사람이 확정한 **장소 목록(gazetteer)**으로 셉니다. "후보는 기계가, 목록은 사람이."
Two steps: (1) a rule proposes **candidates** for you to skim; (2) you count with a **place list (gazetteer)** you decided on. "The machine proposes, the researcher decides."

**프롬프트 / Prompt (EN)**
```text
Using boz (56 sketches), find the places Dickens mentions.
1) Candidates: find capitalised words or phrases (1-3 words, they may contain hyphens like
   Covent-garden) that come right after in, at, to, from, near, towards, through, into
   (optionally followed by "the"). Ignore titles like Mr, Mrs, Miss. Show the 30 most frequent.
2) Count this list of places: PLACES = ["London", "Covent Garden", "Newgate", ...].
   Dickens often writes a hyphen and a lowercase second word ("Covent-garden", "Drury-lane"),
   so treat "-" and " " as the same and accept a lowercase or capitalised second word.
3) Show a table with place, mentions, number of sketches, and draw a horizontal bar chart
   of the top 15. Save it as places.csv.
```
**프롬프트 / Prompt (KO)**
```text
boz(56편)에서 디킨스가 언급한 장소를 찾아 줘.
1) 후보: in, at, to, from, near, towards, through, into 바로 뒤(사이에 "the"가 있어도 됨)에 오는
   대문자로 시작하는 1-3단어 표현(Covent-garden처럼 하이픈 포함 가능)을 찾아. Mr, Mrs, Miss 같은 호칭은 빼고 상위 30개를 보여 줘.
2) 내가 주는 장소 목록 PLACES = ["London", "Covent Garden", "Newgate", ...]을 세어 줘.
   디킨스는 "Covent-garden", "Drury-lane"처럼 하이픈과 소문자를 자주 쓰니까 "-"와 " "를 같게 보고, 두 번째 단어의 대소문자는 둘 다 받아 줘.
3) 장소, 언급 횟수, 등장 작품 수 표를 만들고, 상위 15개를 가로 막대그래프로 그려 줘. places.csv로 저장해.
```
**고치기 프롬프트 / Fix-it prompt:** `"Covent Garden" is counted 0 times, but the text has "Covent-garden". Fix the pattern.`

In [ ]:
# 3-1. 후보 뽑기 / propose candidates
text_all = re.sub(r"\s+", " ", norm("\n".join(boz.text)))          # 줄바꿈도 공백으로 / newlines → spaces
CAND = re.compile(r"\b(?:in|at|to|from|near|towards|through|into)\s+(?:the\s+)?"
                  r"((?:St\. )?[A-Z][a-z']+(?:[- ](?:[A-Z][a-z']+|street|lane|square|garden|gardens|road|"
                  r"yard|hill|bridge|cross|market|fields|inn|park|place|court|row)){0,2})")
SKIP = {"Mr", "Mrs", "Miss", "Sir", "Lord", "I", "The", "No", "Heaven", "God"}
cands = collections.Counter(m for m in CAND.findall(text_all) if m.split()[0].split("-")[0] not in SKIP)
print(f"{len(cands)} different candidates. Top 30:")
print(", ".join(f"{c} ({n})" for c, n in cands.most_common(30)))

In [ ]:
# 3-2. 사람이 정한 장소 목록으로 세기 / count with a place list you decided on
PLACES = ["London", "Covent Garden", "Newgate", "Gravesend", "Scotland Yard", "Astley's", "Holborn",
          "Thames", "Drury Lane", "Ramsgate", "Monmouth Street", "Strand", "Old Bailey", "Islington",
          "Vauxhall", "Margate", "Oxford Street", "Hammersmith", "Camberwell", "Cheapside", "Seven Dials",
          "Greenwich", "Westminster", "Charing Cross", "St. Giles's", "Somers Town", "Pentonville",
          "Whitechapel", "Smithfield", "Temple Bar", "London Bridge", "Blackfriars", "Piccadilly",
          "Regent Street", "Fleet Street", "Bloomsbury", "Paddington", "Kennington", "Clapham",
          "Hampstead", "Highgate", "Richmond", "Brighton", "Battersea", "Lambeth", "Southwark",
          "Doctors' Commons", "St. Paul's", "Haymarket", "Somerset House", "Lincoln's Inn", "Gray's Inn",
          "Long Acre", "Mansion House", "City Road", "Shoreditch", "Walworth", "Kentish Town",
          "Knightsbridge", "Hyde Park", "Regent's Park", "St. James's", "Billingsgate", "Hackney",
          "Twickenham", "Windsor", "Epsom"]

def place_pattern(name):
    """'Covent Garden' -> also matches 'Covent-garden', 'Covent garden', across line breaks."""
    first, *others = re.split(r"[- ]", name)
    others = [f"[{p[0].upper()}{p[0].lower()}]{re.escape(p[1:])}" for p in others]
    return re.compile(r"\b" + r"[-\s]+".join([re.escape(first)] + others) + r"(?![A-Za-z])")

rows = []
for name in PLACES:
    pat = place_pattern(name)
    per_sketch = [len(pat.findall(norm(t))) for t in boz.text]
    if sum(per_sketch):
        rows.append({"place": name, "mentions": sum(per_sketch),
                     "sketches": sum(1 for x in per_sketch if x)})
places = pd.DataFrame(rows).sort_values(["mentions", "sketches"], ascending=False).reset_index(drop=True)
places.index += 1
places.to_csv("places.csv", index_label="rank")
print(f"{len(places)} of {len(PLACES)} listed places found. Top 20:")
print(places.head(20).to_string())

top15 = places.head(15).iloc[::-1]
plt.figure(figsize=(7, 5))
plt.barh(top15.place, top15.mentions, color="#2a78d6")
plt.title("Places in Sketches by Boz (56 sketches)")
plt.xlabel("mentions")
plt.tight_layout()
plt.show()

**무엇을 정했나요? / What did you decide?**
- **목록은 누가 만들었나? / Who made the list?** 목록에 없는 장소는 영원히 0회입니다. 3-1의 후보에서 빠진 장소를 찾아 `PLACES`에 넣어 보세요. / A place not on the list is always 0. Add missing ones from the candidates.
- **'London'을 셀까? / Count "London"?** 가장 많지만 '어디'에 대한 정보는 거의 없습니다. / The most frequent, the least informative.
- **애매한 이름 / Ambiguous names:** `Temple`(법학원) vs temple, `Fleet`(거리·감옥), `the Bank`, `the City`는 목록에서 뺐습니다. 넣는다면 어떻게 구별할까요? / We left these out. How would you disambiguate them?
- **런던 밖 / Outside London:** Gravesend, Ramsgate, Margate(템스 강 하구·해변 휴양지)는 'Boz의 런던'에 넣나요? / Are seaside trips part of "Boz's London"?
- **겹치는 이름 / Nested names:** `Westminster Bridge`를 추가하면 `Westminster`와 두 번 셉니다. / Adding "Westminster Bridge" double-counts "Westminster".
- 참고: 개체명 인식(NER, 예: spaCy)도 쓸 수 있지만, 사전 테스트에서 Tibbs, Agnes 같은 **인물명을 장소로** 분류하기도 했습니다. / NER tools also mislabelled character names (Tibbs, Agnes) as places in our pre-test.

---
## 4. 언급된 거리와 빈도 / Streets mentioned, with frequencies

**연구 질문 / Question:** 어떤 거리가 가장 자주 나오는가? 그리고 디킨스의 표기법이 '세기'를 어떻게 바꾸는가? / Which streets appear most often — and how does Dickens's spelling change the count?

과제 3은 **목록(닫힌 방식)**, 과제 4는 **패턴(열린 방식)**입니다. 패턴은 목록에 없는 거리도 찾지만, 엉뚱한 것도 잡습니다. / Task 3 used a **list** (closed); Task 4 uses a **pattern** (open): it finds unlisted streets, and some junk.

**프롬프트 / Prompt (EN)**
```text
Using boz, find every street-like name and count it.
1) First try a simple regular expression: one capitalised word + a space + Street, Lane, Square
   or Road (like "Oxford Street"). Print the number of matches and the top 10.
2) Dickens usually writes "Monmouth-street", "Drury-lane", "Covent-garden": a hyphen and a
   lowercase suffix. Write a better regular expression that accepts a hyphen or a space and a
   lowercase or capitalised suffix (street, lane, square, road, court, row, place, garden, gardens,
   yard, hill, bridge, fields, walk, market, dials, inn, terrace, buildings, alley, circus, cross, park),
   and allows prefixes like Great, Little, Upper, Lower, Old, New, St.
3) Normalise every match to one spelling ("Monmouth Street"), drop leading words such as
   The, This, A, In, At, Our, His, and print the top 20 with counts, the number of sketches,
   and the original spellings. Save streets.csv.
```
**프롬프트 / Prompt (KO)**
```text
boz에서 거리 이름을 모두 찾아 세어 줘.
1) 먼저 간단한 정규식: 대문자로 시작하는 한 단어 + 공백 + Street, Lane, Square, Road("Oxford Street" 같은). 찾은 개수와 상위 10개를 출력해.
2) 디킨스는 "Monmouth-street", "Drury-lane", "Covent-garden"처럼 하이픈과 소문자 접미사를 주로 써.
   하이픈이나 공백, 소문자나 대문자 접미사(street, lane, square, road, court, row, place, garden, gardens, yard, hill, bridge,
   fields, walk, market, dials, inn, terrace, buildings, alley, circus, cross, park)를 모두 받고,
   Great, Little, Upper, Lower, Old, New, St. 같은 앞말도 허용하는 더 나은 정규식을 써 줘.
3) 찾은 것을 하나의 표기("Monmouth Street")로 통일하고, 앞에 붙은 The, This, A, In, At, Our, His 같은 단어는 떼어 내.
   상위 20개를 횟수, 등장 작품 수, 원문 표기와 함께 출력하고 streets.csv로 저장해.
```
**고치기 프롬프트 / Fix-it prompt:** `"In Oxford Street" was counted as a street called "In Oxford". Strip leading words like In, At, The instead of keeping them.`

In [ ]:
# 4-1. 간단한 정규식 / the naive pattern
naive = re.compile(r"\b([A-Z][a-z]+) (Street|Lane|Square|Road)\b")
naive_hits = collections.Counter(" ".join(m) for m in naive.findall(text_all))
print(f"NAIVE 'Xxx Street': {sum(naive_hits.values())} matches, {len(naive_hits)} different")
print("  ", naive_hits.most_common(10))

In [ ]:
# 4-2. 하이픈·소문자를 받는 정규식 / a pattern that accepts hyphens and lowercase suffixes
SUFFIX = ("street|lane|square|road|court|row|place|garden|gardens|yard|hill|bridge|fields|walk|"
          "market|dials|inn|terrace|buildings|alley|circus|cross|park")
STREET = re.compile(r"\b((?:St\. |Great |Little |Upper |Lower |Old |New )?[A-Z][a-z']+(?:[- ][A-Z][a-z']+)?)"
                    r"[- ]((?i:" + SUFFIX + r")(?:-(?:road|street|lane))?)\b")     # Tottenham-court-road
LEADING = {"The", "This", "That", "A", "An", "Our", "His", "Her", "Their", "Its", "In", "At", "Of", "And",
           "But", "On", "To", "By", "From", "Every", "Each", "Some", "Any", "No", "Which", "Same", "Next",
           "Back", "Front", "High", "Main", "Opposite", "Into", "Through", "Near", "Mr", "Mrs", "Miss"}

def clean_name(name, suffix):
    words = name.replace("-", " ").split()
    while words and words[0] in LEADING:          # 앞말 떼기 / strip leading words
        words = words[1:]
    if not words or words[-1] in LEADING:
        return None
    return (" ".join(words) + " " + suffix.lower().replace("-", " ")).title().replace("'S", "'s")

count, spellings, in_sketch = collections.Counter(), collections.defaultdict(collections.Counter), collections.defaultdict(set)
for title, text in zip(boz.title, boz.text):
    for m in STREET.finditer(re.sub(r"\s+", " ", norm(text))):
        key = clean_name(m.group(1), m.group(2))
        if key:
            count[key] += 1
            spellings[key][m.group(0)] += 1
            in_sketch[key].add(title)

streets = pd.DataFrame([{"street": k, "mentions": n, "sketches": len(in_sketch[k]),
                         "spellings in the text": " | ".join(s for s, _ in spellings[k].most_common(3))}
                        for k, n in count.most_common()])
streets.index += 1
streets.to_csv("streets.csv", index_label="rank")
print(f"HYPHEN-AWARE: {sum(count.values())} matches, {len(count)} different (naive: {sum(naive_hits.values())})")
print(streets.head(20).to_string())

**무엇을 정했나요? / What did you decide?**
- **'거리'의 정의 = 접미사 목록 / Your suffix list *is* your definition of "street".** `Covent Garden`(시장), `Seven Dials`(교차로), `Lincoln's Inn`(법학원), `London Bridge`도 거리인가요? 빼려면 `SUFFIX`에서 지우세요. / Are a market, a junction, an Inn of Court, a bridge "streets"?
- **표기 통일 / Normalising spellings:** `Monmouth-street`과 `Monmouth Street`를 하나로 셌습니다. 덕분에 셀 수 있지만, 19세기 하이픈 표기 관습이라는 정보는 표에서 사라집니다(그래서 '원문 표기' 열을 남겼습니다). / We merged spellings, so we kept the original forms in a column.
- **놓친 것 / What the pattern misses:** `the Strand`, `Cheapside`, `Holborn`처럼 접미사가 없는 거리는 이 방법으로 잡히지 않습니다. 과제 3의 목록과 합쳐야 합니다. / Streets without a suffix need Task 3's list.
- **잘못 잡은 것 / False hits:** 표 아래쪽을 훑어보세요. 거리가 아닌 것이 있나요? / Scan the bottom of the table: any non-streets?

---
## 5. 연어 / Collocations

**연구 질문 / Question:** Boz의 텍스트에서 어떤 두 단어가 우연보다 훨씬 자주 붙어 나오는가? / Which two-word pairs occur together far more often than chance?

- **로그우도비 / Log-likelihood (LLR):** 자주 나오고 확실한 쌍을 위로 올립니다. / Favours frequent, reliable pairs.
- **PMI:** 서로에게만 붙는 쌍을 올립니다. 드문 쌍에 후하므로 최소 빈도가 중요합니다. / Favours exclusive pairs; generous to rare pairs, so a minimum frequency matters.

**프롬프트 / Prompt (EN)**
```text
Using boz, find collocations (two-word pairs that appear together more often than chance).
1) Split all 56 sketches into lowercase words (replace the curly ’ with ').
2) Use nltk.collocations.BigramCollocationFinder. Remove pairs that contain an NLTK stopword
   or a word shorter than 3 letters, and keep pairs that appear at least 5 times.
3) Rank the pairs by log-likelihood ratio and by PMI and show the top 20 of each side by side,
   with their frequencies.
4) Many top pairs are character names (e.g. "watkins tottle"). Find words that are almost always
   capitalised in the middle of a sentence (likely names), remove them, and show the top 20 again.
```
**프롬프트 / Prompt (KO)**
```text
boz에서 연어(우연보다 자주 붙어 나오는 두 단어 쌍)를 찾아 줘.
1) 56편 전체를 소문자 단어로 나눠(곡선 ’는 '로).
2) nltk.collocations.BigramCollocationFinder를 써. NLTK 불용어나 3글자 미만 단어가 들어간 쌍은 빼고, 5번 이상 나온 쌍만 남겨.
3) 로그우도비와 PMI로 각각 순위를 매겨 상위 20개를 빈도와 함께 나란히 보여 줘.
4) 상위 쌍 상당수가 인물 이름이야(예: "watkins tottle"). 문장 중간에서 거의 항상 대문자로 쓰이는 단어(이름일 가능성이 큼)를 찾아 빼고 다시 상위 20개를 보여 줘.
```
**고치기 프롬프트 / Fix-it prompt:** `The PMI list shows only pairs that appear once or twice. Apply a minimum frequency of 5 before ranking.`

In [ ]:
# 5-1. 이름을 그대로 둔 연어 / collocations with names kept
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
measures = BigramAssocMeasures()

def collocations(drop=frozenset(), n=20, min_freq=5):
    finder = BigramCollocationFinder.from_words(all_words)          # 원래 순서의 인접 쌍 / adjacent pairs
    finder.apply_word_filter(lambda w: w in NLTK_SW or len(w) < 3 or w in drop)
    finder.apply_freq_filter(min_freq)
    fd = finder.ngram_fd
    fmt = lambda pairs: [f"{a} {b} ({fd[(a, b)]})" for a, b in pairs]
    return pd.DataFrame({"log-likelihood": fmt(finder.nbest(measures.likelihood_ratio, n)),
                         "PMI": fmt(finder.nbest(measures.pmi, n))}, index=range(1, n + 1))

print("Names KEPT (min. frequency 5):")
print(collocations().to_string())

In [ ]:
# 5-2. 이름으로 보이는 단어 찾기 / detect likely names
# 문장 중간(소문자나 쉼표 뒤)에서 90% 이상 대문자로 쓰인 단어 / capitalised >90% of the time mid-sentence
cap, low = collections.Counter(), collections.Counter()
for w in re.findall(r"(?<=[a-z,;] )([A-Za-z](?:[a-z]|'[a-z])+)", text_all):    # Tibbs, J'mima
    (cap if w[0].isupper() else low)[w.lower()] += 1
NAMES = {w for w in cap if cap[w] >= 3 and cap[w] / (cap[w] + low[w]) > 0.9
         and not w.startswith("i'")} - WHITELIST                     # I'm, I'll 은 이름이 아님
print(f"{len(NAMES)} likely names, most frequent 40:")
print(", ".join(sorted(NAMES, key=lambda w: -cap[w])[:40]))

print("\nNames REMOVED:")
print(collocations(drop=NAMES).to_string())

**무엇을 정했나요? / What did you decide?**
- **이름을 뺄까? / Remove names?** '사람' 또는 '관계'가 질문이라면 이름이 곧 데이터입니다(다음 수업의 네트워크). 장면·사물이 질문이라면 이름은 잡음입니다. / Names are noise for "things and scenes" but data for "people and relations".
- **자동 이름 목록 점검 / Check the automatic name list:** `london`, `sunday` 같은 단어도 들어갔나요? 그대로 둘까요? 규칙(90%, 3회 이상)도 여러분의 선택입니다. / Did place or day names sneak in? The 90% / 3-times rule is your choice too.
- **LLR vs PMI, 최소 빈도 5:** 바꾸면 무엇이 달라지나요? (`collocations(min_freq=10)`) / Try other thresholds.
- `hackney coach`(삯마차)처럼 1830년대 런던의 교통이 보이나요? 어떤 쌍이 '연구할 만한' 쌍인가요? / Do you see 1830s transport? Which pairs are worth a research question?

---
## 6. 토픽 모델링 / Topic modelling

**연구 질문 / Question:** 56편 안에 어떤 주제 묶음이 흐르는가? 그리고 인물 이름이 그 답을 어떻게 바꾸는가? / What thematic clusters run through the 56 sketches — and how do character names change the answer?

- **300단어 조각 / 300-word chunks:** 56편은 문서 수로는 적고 길이 차도 큽니다. 그래서 각 편을 300단어씩 잘라 문서로 씁니다. / 56 documents are too few and too uneven, so we cut each sketch into 300-word chunks.
- **NMF**(TF-IDF 기반)를 기본으로 쓰고, 마지막에 **LDA**도 선택으로 돌려 봅니다. / NMF by default; LDA optional.

**프롬프트 / Prompt (EN)**
```text
Using boz, do topic modelling with scikit-learn.
1) Split each sketch into chunks of 300 words (drop chunks shorter than 100 words) and remember
   which sketch each chunk came from.
2) Use TfidfVectorizer (NLTK English stopwords plus my extra list, max_df=0.5, min_df=3) and
   NMF with 8 topics (random_state=0).
3) For each topic print the top 10 words and the 2 sketches that contribute most chunks to it.
4) Run it twice: first as is, then after also removing likely character names (words that are
   almost always capitalised in the middle of a sentence). Keep the runtime under one minute.
```
**프롬프트 / Prompt (KO)**
```text
boz로 scikit-learn 토픽 모델링을 해 줘.
1) 각 작품을 300단어 조각으로 나누고(100단어보다 짧은 조각은 버림), 조각마다 어느 작품에서 왔는지 기록해.
2) TfidfVectorizer(NLTK 영어 불용어 + 내 추가 목록, max_df=0.5, min_df=3)와 토픽 8개짜리 NMF(random_state=0)를 써.
3) 토픽마다 상위 10개 단어와, 그 토픽에 조각을 가장 많이 준 작품 2편을 출력해.
4) 두 번 돌려 줘: 처음엔 그대로, 다음엔 인물 이름(문장 중간에서 거의 항상 대문자인 단어)도 뺀 뒤. 실행 시간은 1분 안으로.
```
**고치기 프롬프트 / Fix-it prompt:** `Every topic is just the names of one story's characters. Remove the likely names and run it again.`

In [ ]:
# 6-1. 300단어 조각 만들기 / make 300-word chunks
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

CHUNK = 300
chunks, chunk_title = [], []
for title, text in zip(boz.title, boz.text):
    w = tokens(text)
    for k in range(0, len(w), CHUNK):
        piece = w[k:k + CHUNK]
        if len(piece) >= 100:                   # 너무 짧은 꼬리 조각은 버림 / drop short tails
            chunks.append(" ".join(piece))
            chunk_title.append(title)
print(f"{len(chunks)} chunks of up to {CHUNK} words from {len(boz)} sketches")

# 토픽용 불용어: 여기서는 호칭도 뺍니다(과제 2와 다른 선택) / here we DO drop titles (a different choice from Task 2)
TOPIC_STOP = NLTK_SW | WHITELIST | {"said", "would", "could", "one", "upon", "much", "little", "two",
                                    "well", "may", "might", "must", "shall", "never", "every", "like"}

def show_topics(model, terms, W, n_words=10):
    for i, comp in enumerate(model.components_):
        words = " ".join(terms[j] for j in comp.argsort()[::-1][:n_words])
        src = collections.Counter(chunk_title[d] for d in range(len(chunks)) if W[d].argmax() == i)
        src = ", ".join(f"{t[:32]} ({n})" for t, n in src.most_common(2))
        print(f" T{i}: {words}\n      <- {src}")

def run_nmf(extra_stop=frozenset(), k=8):
    t0 = time.time()
    vec = TfidfVectorizer(stop_words=sorted(TOPIC_STOP | set(extra_stop)), max_df=0.5, min_df=3,
                          token_pattern=r"[a-z]+(?:'[a-z]+)*")
    X = vec.fit_transform(chunks)
    model = NMF(n_components=k, random_state=0, init="nndsvda", max_iter=500)
    W = model.fit_transform(X)
    show_topics(model, vec.get_feature_names_out(), W)
    print(f" ({time.time() - t0:.1f} s, {X.shape[1]:,} words in the vocabulary)")

In [ ]:
# 6-2. 이름을 둔 경우 vs 뺀 경우 / names kept vs names removed
print("== NMF, 8 topics, names KEPT ==")
run_nmf()
print("\n== NMF, 8 topics, names REMOVED ==")
run_nmf(extra_stop=NAMES)

In [ ]:
# 6-3. (선택 / optional) LDA로 비교 / compare with LDA (word counts instead of TF-IDF)
t0 = time.time()
vec = CountVectorizer(stop_words=sorted(TOPIC_STOP | NAMES), max_df=0.5, min_df=3,
                      token_pattern=r"[a-z]+(?:'[a-z]+)*")
X = vec.fit_transform(chunks)
lda = LatentDirichletAllocation(n_components=8, random_state=0, learning_method="batch", max_iter=20)
W = lda.fit_transform(X)
print("== LDA, 8 topics, names REMOVED ==")
show_topics(lda, vec.get_feature_names_out(), W)
print(f" ({time.time() - t0:.1f} s)")

**무엇을 정했나요? / What did you decide?**
- **이름 / Names:** 이름을 두면 토픽이 대부분 '이야기 한 편의 등장인물 목록'이 됩니다. 빼면 교통(coach, cab, omnibus), 식사와 건배, 가난과 죽음 같은 주제가 드러납니다. / With names, a topic is mostly one story's cast; without them, themes appear.
- **NMF vs LDA:** 이 정도 크기(조각 629개)에서는 LDA 토픽이 NMF보다 흐릿하게 나옵니다. 어느 쪽 결과를 믿을지, 어떻게 확인할지도 선택입니다. / On a corpus this small, LDA topics come out blurrier than NMF's. Which one do you trust, and how would you check?
- **조각 크기와 토픽 수 / Chunk size and k:** `CHUNK`를 100이나 1000으로, `run_nmf(k=5)`나 `k=12`로 바꿔 보세요. 어떤 설정이 '해석할 만한가'? / Try other values. Which setting is interpretable?
- **토픽 이름 붙이기 / Naming topics:** 상위 단어에 이름을 붙이는 것은 해석입니다. AI에게 이름을 제안받을 수 있지만, 원문 조각을 읽고 검증하는 것은 연구자의 몫입니다. / An AI can propose labels; checking them against the chunks is your job.
- **참고 / Note:** 문서 1만 개 기준 LDA는 몇 분·무료, LLM 기반 TopicGPT는 9시간·$65였다는 비교가 있습니다(Li et al., ACL 2025). 고전적인 방법이 여전히 빠르고 싼 출발점입니다. / Classic topic models remain a fast, cheap baseline.

---
## 내 결과 내려받기 / Download your results

왼쪽의 폴더 아이콘(Files)을 누르면 `boz_56.csv`, `places.csv`, `streets.csv`가 보입니다. 파일 이름 옆 ⋮ → Download. 세션이 끝나면 지워지니 필요하면 내려받으세요.
Click the folder icon (Files) on the left to find `boz_56.csv`, `places.csv`, `streets.csv` (⋮ → Download). They are deleted when the session ends.

In [ ]:
for f in ["boz_56.csv", "places.csv", "streets.csv"]:
    print(f"{f:12s} {os.path.getsize(f):>10,} bytes")

---
## 기말 프로젝트를 위한 다음 단계 / Next steps for your final project

- **네트워크 / Network → 다음 수업 / next class.** 과제 5·6의 `NAMES` 목록이 인물(노드) 후보입니다. '관계'를 무엇으로 정의할지(같은 문단에 등장? 대화? 시중듦?)부터 정해 두세요. / `NAMES` gives candidate nodes; decide first what counts as a relation.
- **지도 / Mapping → 다음 수업 / next class.** `places.csv`와 `streets.csv`가 지도의 입력입니다. 좌표는 사람이 확인합니다(자동 지오코딩은 Margate를 100 km 넘게 엉뚱한 곳에 찍기도 했습니다). / Your CSVs are the input; coordinates must be checked by a person.
- **더 큰 코퍼스 / A bigger corpus.** 같은 코드로 디킨스 장편을 더하면 토픽 모델링이 더 안정됩니다. Gutenberg 번호를 바꿔 내려받으세요. / Add Dickens's novels and re-run Task 6:
  *The Pickwick Papers* 580 · *Oliver Twist* 730 · *David Copperfield* 766 · *Bleak House* 1023 · *A Tale of Two Cities* 98 · *Great Expectations* 1400 · *Our Mutual Friend* 883
  (URL: `https://www.gutenberg.org/cache/epub/<번호>/pg<번호>.txt` — 한 권에 한 번만 요청하세요 / one request per book)
  - 주의: 장편은 'CHAPTER' 형식과 앞뒤 구성이 다릅니다. 과제 1의 나누기 코드를 고쳐야 합니다 — 좋은 바이브 코딩 연습입니다. / Each novel's layout differs, so you will need to adapt Task 1 — good vibe-coding practice.
- **연구노트 / Research notes.** 각 과제의 "무엇을 정했나요?"에 대한 답을 모으면 방법론 절(methods section)의 초안이 됩니다. / Your answers to "What did you decide?" are the draft of your methods section.